# DS2002 · Capstone Kickoff and First Pulls

**Studio — 2026-11-18 · Fall 2026**
**Class time:** 45 minutes

---

## Capstone kickoff and first pulls

The capstone starts today. Read `DS2002 Capstone — ACC Reality Check.pdf` in this folder before Friday, because it describes everything your team will hand in and when. The short version is that your team will decide which ACC men's basketball teams are as good as their records say and which ones have built their records on easy opponents. To do that you will combine a ratings file with game results from a live API, keep everything in a SQLite database, answer the questions with SQL, and publish your results as a small web API.

Today has a modest goal. Each team forms, sets up a repository, and looks at both sources with its own eyes. You will not clean or load anything yet. The point of the hour is to find out what the data looks like before anyone writes a plan around it.

If you do not follow basketball, start with the appendix of the PDF. It explains every term used in this notebook, and nobody on your team should feel behind because of the sport.

### Step 1: Form your team

Teams have three or four people. Fill in the cell below together and commit it to the team repository by the end of class. The roles are a starting point, and you can trade them later, but someone should own each part of the project from the first day.

In [1]:
team = {
    'name': 'TODO',
    'members': ['Megha', 'Hershi', 'Angelia'],
    'repo_url': 'TODO',
    'roles': {
        'ratings CSV and crosswalk': 'Megha',
        'ESPN pulls and box scores': 'Hershi',
        'SQLite schema and queries': 'Angelia',
        'API and PythonAnywhere': 'Hershi',
    },
    'meeting_time': 'TODO',
}

unfilled = [k for k, v in team.items() if 'TODO' in str(v)]
print('still unfilled:', unfilled if unfilled else 'nothing, the charter is complete')

still unfilled: ['name', 'repo_url', 'meeting_time']


### Step 2: Pull the ratings CSV

Bart Torvik publishes a rating for every Division I team as a CSV file at this address:

`https://barttorvik.com/2027_team_results.csv`

The 2027 in the name refers to the season that ends in 2027, which is the season being played right now. You do not need a key or an account.

**TODO:** download the file with `requests`, read it into pandas, and look at it before you do anything else. Print the shape, the column names, and the first few rows. Save the raw text to a file named with today's date, because the ratings change every day and you will want to know which version you used.

In [2]:
import io
from datetime import date
from pathlib import Path

import pandas as pd
import requests

TORVIK_URL = 'https://barttorvik.com/2027_team_results.csv'

raw_dir = Path('data/raw')
raw_dir.mkdir(parents=True, exist_ok=True)
raw_path = raw_dir / f'torvik_2027_team_results_{date.today().isoformat()}.csv'

# The PDF asks us to save every raw response and read the saved copy when it exists,
# so we only download if today's snapshot is not already on disk.
if raw_path.exists():
    print('reading saved snapshot', raw_path)
    raw_text = raw_path.read_text(encoding='utf-8')
else:
    resp = requests.get(TORVIK_URL, timeout=30)
    print('status:', resp.status_code, '| content-type:', resp.headers.get('content-type'))
    resp.raise_for_status()
    raw_text = resp.text
    raw_path.write_text(raw_text, encoding='utf-8')   # raw text exactly as downloaded
    print('saved raw file to', raw_path)

torvik = pd.read_csv(io.StringIO(raw_text))
print('shape:', torvik.shape)
print('columns:', list(torvik.columns))
torvik.head()

reading saved snapshot data/raw/torvik_2027_team_results_2026-11-19.csv
shape: (365, 45)
columns: ['rank', 'team', 'conf', 'record', 'adjoe', 'oe Rank', 'adjde', 'de Rank', 'barthag', 'rank.1', 'proj. W', 'Proj. L', 'Pro Con W', 'Pro Con L', 'Con Rec.', 'sos', 'ncsos', 'consos', 'Proj. SOS', 'Proj. Noncon SOS', 'Proj. Con SOS', 'elite SOS', 'elite noncon SOS', 'Opp OE', 'Opp DE', 'Opp Proj. OE', 'Opp Proj DE', 'Con Adj OE', 'Con Adj DE', 'Qual O', 'Qual D', 'Qual Barthag', 'Qual Games', 'FUN', 'ConPF', 'ConPA', 'ConPoss', 'ConOE', 'ConDE', 'ConSOSRemain', 'Conf Win%', 'WAB', 'WAB Rk', 'Fun Rk', 'adjt']


,rank,team,conf,record,adjoe,oe Rank,adjde,de Rank,barthag,rank.1,...,ConPA,ConPoss,ConOE,ConDE,ConSOSRemain,Conf Win%,WAB,WAB Rk,Fun Rk,adjt
0,1,Kansas,B12,4-0,121.660451,6,91.921735,2,0.961706,1,...,0.0,0.0,0,0,0.842296,0,1.019160,9,106,72.421551
1,2,Houston,B12,5-0,122.204510,5,93.181538,5,0.957637,2,...,0.0,0.0,0,0,0.846314,0,1.581425,2,57,67.515145
2,3,Duke,ACC,3-2,119.702453,14,91.900404,1,0.954327,3,...,0.0,0.0,0,0,0.804628,0,-0.447576,126,353,72.011265
3,4,Illinois,B10,4-1,124.442649,1,96.269347,18,0.950358,4,...,0.0,0.0,0,0,0.840569,0,0.961320,10,126,74.486749
4,5,Florida,SEC,5-1,122.968410,2,95.187414,10,0.950023,5,...,0.0,0.0,0,0,0.860444,0,1.737383,1,101,69.678265


In [3]:
# Q1: how many rows belong to the ACC? (filter on the 'conf' column)
acc = torvik[torvik['conf'] == 'ACC']
print('total rows:', len(torvik), '| ACC rows:', len(acc))
acc[['rank', 'team', 'conf', 'record', 'barthag', 'rank.1']]

total rows: 365 | ACC rows: 18


,rank,team,conf,record,barthag,rank.1
2,3,Duke,ACC,3-2,0.954327,3
5,6,Virginia,ACC,6-0,0.945065,6
15,16,Miami FL,ACC,4-1,0.920536,16
17,18,Louisville,ACC,2-1,0.919407,18
32,33,N.C. State,ACC,4-0,0.880878,33
35,36,Virginia Tech,ACC,5-0,0.874377,36
48,49,North Carolina,ACC,3-1,0.847074,49
51,52,Florida St.,ACC,4-1,0.835871,52
61,62,SMU,ACC,5-0,0.822757,62
68,69,Notre Dame,ACC,5-1,0.811948,69


In [4]:
# Q2: the header 'rank' appears twice. pandas keeps the first as 'rank' and renames the second 'rank.1'.
# Do they agree, and does either match an ordering by barthag?
print('rank == rank.1 for every row:', (torvik['rank'] == torvik['rank.1']).all())
barthag_order = torvik['barthag'].rank(ascending=False, method='min').astype(int)
print('rank.1 matches barthag order:', (barthag_order == torvik['rank.1']).all())

rank == rank.1 for every row: True
rank.1 matches barthag order: True


In [5]:
# Q3: spellings of a few schools exactly as Torvik writes them
for word in ['State', 'Miami', 'Florida St', 'Pitt', 'Corpus', 'Ball St']:
    print(f'{word!r:14}', torvik.loc[torvik['team'].str.contains(word), 'team'].tolist())

'State'        ['N.C. State']
'Miami'        ['Miami FL', 'Miami OH']
'Florida St'   ['Florida St.']
'Pitt'         ['Pittsburgh']
'Corpus'       ['Texas A&M Corpus Chris']
'Ball St'      ['Ball St.']


**TODO:** answer these in the cell below, using what you printed.

1. How many rows are there, and how many belong to the ACC? You should be able to filter on one column to find out.
2. Two of the original headers have the same name. What did pandas call them, and which one would you trust as the team's rank?
3. Look at how a few schools are spelled, such as North Carolina State and Miami. Write the spellings down exactly, because the other source will spell them differently.

**1.** The file has **365 rows** (one per Division I team) and 45 columns. Filtering on `conf == 'ACC'` gives **18 ACC teams**.

**2.** Both headers are named `rank`. pandas kept the first as **`rank`** and renamed the second **`rank.1`**. In today's file the two are identical for all 365 rows. Both also match ordering the teams by `barthag`, Torvik's overall rating, from highest to lowest. I would trust **`rank.1`** as the team's rank because it sits directly after `barthag`, which shows it is the rank of that rating. The first `rank` looks like the row position in a file that happens to be sorted by barthag. When we clean the data, we'll rename `rank.1` to `barthag_rank` and drop the other one, after checking that they still agree.

**3.** Torvik spellings, copied exactly:
- North Carolina State → **`N.C. State`**
- Miami (the ACC school) → **`Miami FL`**. Note that `Miami OH` also exists, so a loose "contains Miami" match would be wrong.
- Florida State → **`Florida St.`** (Torvik abbreviates "State" to "St." with a period: `Ball St.`, `Georgia St.`)
- Pittsburgh → **`Pittsburgh`**
- Texas A&M–Corpus Christi → **`Texas A&M Corpus Chris`** (truncated, with no hyphen)

### Step 3: Pull one day of games from ESPN

ESPN serves game results as JSON with no key. This address returns every game an ACC team played on one date:

`https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/scoreboard`

Pass the parameters `groups=2` (the ACC), `dates=YYYYMMDD`, and `limit=200`. Pick a date from the past week. You practiced this kind of request in October, so reuse what worked for you then, including a timeout and a check of the status code.

**TODO:** make the request, then explore the JSON one level at a time. Find the list of games, count them, and for one game print both team names, both scores, whether the game is finished, and the start time exactly as ESPN wrote it.

In [6]:
import json

ESPN_URL = 'https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/scoreboard'
GAME_DATE = '20261113'   # Friday, Nov 13, 2026

espn_path = raw_dir / f'espn_scoreboard_acc_{GAME_DATE}.json'

if espn_path.exists():
    print('reading saved response', espn_path)
    scoreboard = json.loads(espn_path.read_text(encoding='utf-8'))
else:
    r = requests.get(ESPN_URL, params={'groups': 2, 'dates': GAME_DATE, 'limit': 200}, timeout=30)
    print('status:', r.status_code)
    if r.status_code != 200:
        raise RuntimeError(f'ESPN request failed: {r.status_code}')
    espn_path.write_text(r.text, encoding='utf-8')     # raw response exactly as received
    scoreboard = r.json()

# ESPN does not document this API, so fail loudly if the shape changes
assert 'events' in scoreboard, 'ESPN response has no "events" key'

# Level 1: top-level keys
print('top-level keys:', list(scoreboard.keys()))

status: 200
top-level keys: ['leagues', 'groups', 'events', 'provider', 'eventsDate']


In [7]:
# Level 2: the games live in scoreboard['events']
events = scoreboard['events']
print('number of games:', len(events))

game = events[0]
print('keys in one event:', list(game.keys()))

# Level 3: each event has a list 'competitions' (always one game here)
comp = game['competitions'][0]
print('keys in competition:', list(comp.keys()))

# Level 4: the two teams are in comp['competitors']
print('keys in one competitor:', list(comp['competitors'][0].keys()))
print('keys in competitor["team"]:', list(comp['competitors'][0]['team'].keys()))

number of games: 11
keys in one event: ['id', 'uid', 'date', 'name', 'shortName', 'season', 'competitions', 'links', 'status']
keys in competition: ['id', 'uid', 'date', 'attendance', 'type', 'timeValid', 'neutralSite', 'conferenceCompetition', 'playByPlayAvailable', 'recent', 'venue', 'competitors', 'notes', 'status', 'broadcasts', 'format', 'startDate', 'broadcast', 'geoBroadcasts', 'highlights', 'headlines']
keys in one competitor: ['id', 'uid', 'type', 'order', 'homeAway', 'winner', 'team', 'score', 'linescores', 'statistics', 'leaders', 'curatedRank', 'records']
keys in competitor["team"]: ['id', 'uid', 'location', 'name', 'abbreviation', 'displayName', 'shortDisplayName', 'color', 'alternateColor', 'isActive', 'venue', 'links', 'logo', 'logoDark', 'conferenceId']


In [8]:
# One game: both team names, both scores, finished?, start time as ESPN wrote it
print('game:', game['name'])
for c in comp['competitors']:
    print(f"  {c['homeAway']:4}  {c['team']['displayName']:28} score={c['score']!r}")
print('finished:', game['status']['type']['completed'], '|', game['status']['type']['description'])
print('start time (raw):', game['date'])

game: SIU Edwardsville Cougars at Duke Blue Devils
  home  Duke Blue Devils             score='90'
  away  SIU Edwardsville Cougars     score='47'
finished: True | Final
start time (raw): 2026-11-14T01:00Z


In [9]:
# All games on that date in one table, to answer the questions below
rows = []
for e in events:
    c = e['competitions'][0]
    home = next(t for t in c['competitors'] if t['homeAway'] == 'home')
    away = next(t for t in c['competitors'] if t['homeAway'] == 'away')
    rows.append({
        'event_id': e['id'],
        'start_raw': e['date'],
        'home': home['team']['location'], 'home_id': home['team']['id'], 'home_score': home['score'],
        'away': away['team']['location'], 'away_id': away['team']['id'], 'away_score': away['score'],
        'completed': e['status']['type']['completed'],
        'neutral': c['neutralSite'],
    })
games = pd.DataFrame(rows)

# Compare naive date (first 10 chars) with the real local date in US Eastern time
games['naive_date'] = games['start_raw'].str[:10]
start_utc = pd.to_datetime(games['start_raw'], utc=True)
games['start_eastern'] = start_utc.dt.tz_convert('America/New_York').dt.strftime('%Y-%m-%d %I:%M %p')
games['eastern_date'] = start_utc.dt.tz_convert('America/New_York').dt.date.astype(str)
print('score dtype:', type(games.loc[0, 'home_score']))
print('games whose naive date is wrong:', (games['naive_date'] != games['eastern_date']).sum(), 'of', len(games))
games[['start_raw', 'naive_date', 'start_eastern', 'home', 'home_score', 'away', 'away_score', 'completed']]

score dtype: <class 'str'>
games whose naive date is wrong: 7 of 11


,start_raw,naive_date,start_eastern,home,home_score,away,away_score,completed
0,2026-11-14T01:00Z,2026-11-14,2026-11-13 08:00 PM,Duke,90,SIU Edwardsville,47,True
1,2026-11-14T01:00Z,2026-11-14,2026-11-13 08:00 PM,Maryland,83,Virginia,87,True
2,2026-11-13T22:00Z,2026-11-13,2026-11-13 05:00 PM,Miami,94,North Alabama,52,True
3,2026-11-13T22:00Z,2026-11-13,2026-11-13 05:00 PM,Florida State,84,Texas A&M-Corpus Christi,60,True
4,2026-11-13T23:00Z,2026-11-13,2026-11-13 06:00 PM,Clemson,78,Georgetown,85,True
5,2026-11-13T23:30Z,2026-11-13,2026-11-13 06:30 PM,Butler,64,Pittsburgh,76,True
6,2026-11-14T00:00Z,2026-11-14,2026-11-13 07:00 PM,Virginia Tech,82,Mount St. Mary's,62,True
7,2026-11-14T00:00Z,2026-11-14,2026-11-13 07:00 PM,NC State,81,Troy,62,True
8,2026-11-14T00:00Z,2026-11-14,2026-11-13 07:00 PM,North Carolina,70,Georgia,78,True
9,2026-11-14T02:00Z,2026-11-14,2026-11-13 09:00 PM,UNLV,76,Stanford,68,True


In [10]:
# How ESPN spells the schools from Step 2 (several name fields are available)
for e in events:
    for c in e['competitions'][0]['competitors']:
        t = c['team']
        if t['location'] in ('NC State', 'Miami', 'Florida State', 'Pittsburgh', 'Texas A&M-Corpus Christi'):
            print(f"id={t['id']:>4}  location={t['location']!r:28} displayName={t['displayName']!r:38} short={t['shortDisplayName']!r}")

id=2390  location='Miami'                      displayName='Miami Hurricanes'                     short='Miami'
id=  52  location='Florida State'              displayName='Florida State Seminoles'              short='Florida St'
id= 357  location='Texas A&M-Corpus Christi'   displayName='Texas A&M-Corpus Christi Islanders'   short='Texas A&M-CC'
id= 221  location='Pittsburgh'                 displayName='Pittsburgh Panthers'                  short='Pitt'
id= 152  location='NC State'                   displayName='NC State Wolfpack'                    short='NC State'


In [11]:
# Q4: is every opponent Division I? Torvik lists all 365 D-I teams, so check each opponent against it.
# A quick-and-dirty name match for now ('State' -> 'St.'), and the misses get checked by hand.
acc_espn = {'Duke', 'Virginia', 'Miami', 'Florida State', 'Clemson', 'Pittsburgh', 'Virginia Tech',
            'NC State', 'North Carolina', 'Stanford', 'SMU'}
torvik_names = set(torvik['team'])
for _, g in games.iterrows():
    opp = g['away'] if g['home'] in acc_espn else g['home']
    guess = opp if opp in torvik_names else opp.replace('State', 'St.')
    conf = torvik.loc[torvik['team'] == guess, 'conf'].tolist()
    print(f'{opp:28} -> torvik: {guess if conf else "NO MATCH":24} conf: {conf}')

SIU Edwardsville             -> torvik: SIU Edwardsville         conf: ['OVC']
Maryland                     -> torvik: Maryland                 conf: ['B10']
North Alabama                -> torvik: North Alabama            conf: ['UAC']
Texas A&M-Corpus Christi     -> torvik: NO MATCH                 conf: []
Georgetown                   -> torvik: Georgetown               conf: ['BE']
Butler                       -> torvik: Butler                   conf: ['BE']
Mount St. Mary's             -> torvik: Mount St. Mary's         conf: ['Metro']
Troy                         -> torvik: Troy                     conf: ['SB']
Georgia                      -> torvik: Georgia                  conf: ['SEC']
UNLV                         -> torvik: UNLV                     conf: ['MWC']
Ball State                   -> torvik: Ball St.                 conf: ['MAC']


**TODO:** answer these in the cell below.

1. Where in the JSON are the games, and where inside one game are the two teams?
2. Compare the start time ESPN gives for an evening game with the date you asked for. What time zone is the start time in, and what goes wrong if you use its first ten characters as the game date?
3. How does ESPN spell the schools you wrote down in Step 2?
4. Did every opponent on your date play in Division I? Look the unfamiliar names up.

**1.** The games are the list at **`scoreboard['events']`**, one dict per game. We asked for 2026-11-13 and got **11 games**. Inside one game, the teams are at **`event['competitions'][0]['competitors']`**, a list of two dicts. Each competitor has `homeAway` ('home'/'away'), `score` (a **string** such as `'90'`), `winner`, and a nested `team` dict with `id`, `location`, `displayName`, `shortDisplayName`, and `abbreviation`. Whether the game is finished is at `event['status']['type']['completed']`, and the start time is `event['date']`.

**2.** We asked for **November 13**, but Duke vs. SIU Edwardsville shows `'2026-11-14T01:00Z'`. The trailing **`Z` means UTC** (Zulu time), which is 5 hours ahead of Eastern in November. So 01:00Z on the 14th is **8:00 PM Eastern on Friday the 13th**. If we take the first ten characters, every game that started at 7 PM Eastern or later gets dated **one day late**. On this date that was **7 of the 11 games**, so it is the usual case, not a rare edge case. The fix is to parse the time as UTC, convert to `America/New_York` (or the venue's local time), and only then take the date.

**3.** ESPN spellings (from `team['location']`; `displayName` adds the mascot):

| School | Torvik | ESPN `location` | ESPN `shortDisplayName` | ESPN id |
|---|---|---|---|---|
| NC State | `N.C. State` | `NC State` | `NC State` | 152 |
| Miami | `Miami FL` | `Miami` | `Miami` | 2390 |
| Florida State | `Florida St.` | `Florida State` | `Florida St` (no period) | 52 |
| Pittsburgh | `Pittsburgh` | `Pittsburgh` | `Pitt` | 221 |
| Texas A&M–CC | `Texas A&M Corpus Chris` | `Texas A&M-Corpus Christi` | `Texas A&M-CC` | 357 |

**4.** Yes, on November 13 every opponent was Division I. All 11 opponents appear in Torvik's list of 365 D-I teams. The quick name match missed only one, **Texas A&M–Corpus Christi**, and looking it up by hand shows it is a D-I Southland school that Torvik spells `Texas A&M Corpus Chris` (conf `Slnd`). The unfamiliar names are all D-I: **SIU Edwardsville** (Ohio Valley), **North Alabama** (UAC), **Mount St. Mary's** (MAAC, `Metro` in Torvik), **Troy** (Sun Belt), and **Ball State** (MAC). Not every date will be like this. Early-season schedules often include non-D-I schools (Division II, NAIA), which will have no Torvik row, so the pipeline has to handle opponents that don't match.

### Step 4: Write down the work ahead

You have now seen both sources. Before you leave, list at least five specific things that will have to be cleaned, converted, or matched before the two sources can be joined, and make each one concrete. "Names are different" is less useful than "ESPN writes NC State and Torvik writes N.C. State." This list becomes the first entries in your decision log on Friday.

_1._ **School names need a crosswalk keyed on ESPN's team id.** ESPN writes `NC State` and Torvik writes `N.C. State`. ESPN writes `Miami` and Torvik writes `Miami FL`, and Torvik also has `Miami OH`, so a fuzzy match could pick the wrong school. The crosswalk table should map ESPN `team.id` (e.g. 152, 2390) to the exact Torvik `team` string, because ids don't change spelling.

_2._ **"State" vs. "St." and truncated names.** ESPN `location` says `Florida State` and `Ball State`, while Torvik says `Florida St.` and `Ball St.`. ESPN's `shortDisplayName` says `Florida St` with no period, so we must pick one ESPN field (`location`) and use only that one. Torvik also truncates long names: ESPN `Texas A&M-Corpus Christi` is Torvik `Texas A&M Corpus Chris`. Rules alone won't catch every case, so the crosswalk needs hand-checked rows.

_3._ **Game dates must be converted from UTC before taking the date.** ESPN's `event['date']` is UTC (`'2026-11-14T01:00Z'` for an 8 PM Eastern game on Nov 13). Using the first 10 characters misdated 7 of 11 games on our test date. We'll parse with `utc=True`, convert to `America/New_York`, then store `game_date`.

_4._ **Scores are strings and unfinished games come back too.** ESPN returns `score` as `'90'`, not `90`, so cast it to int. Pulling 2026-11-18 while games were still being played returned Syracuse–Cornell at `STATUS_HALFTIME` with partial scores. Only keep games where `status.type.completed` is `True`, and re-pull days that weren't final.

_5._ **Torvik's duplicate `rank` header and text records.** pandas reads the two `rank` headers as `rank` and `rank.1`. Rename `rank.1` to `barthag_rank`, check that it equals `rank`, and drop the duplicate. `record` (`'3-2'`) and `Con Rec.` (`'0-0'`) are text and need splitting into integer wins/losses columns. Column names have spaces and periods (`proj. W`, `Con Rec.`) that should become snake_case before they go into SQLite.

_6._ **Torvik ratings change every day, so each load needs a snapshot date.** Store the raw file as `torvik_2027_team_results_YYYY-MM-DD.csv` and add a `snapshot_date` column, so every query result can say which ratings it used.

_7._ **Opponents that aren't Division I or aren't ACC.** `groups=2` returns any game with an ACC team in it, so non-ACC teams appear too (e.g. Maryland, Big Ten, vs. Virginia). Non-D-I opponents will have no Torvik row at all. The schema must allow an opponent with no rating, and decide how to treat those games in strength-of-schedule analysis.

_8._ **Home/away doesn't always mean home court, and games can repeat.** `competitions[0].neutralSite` is `True` for some games (Notre Dame vs. Florida on Nov 17), even though ESPN still labels one team "home". An ACC vs. ACC game can also appear in more than one pull, so dedupe on ESPN `event['id']`.

### Before Friday

Friday's check-in notebook walks your team through getting the first version of the pipeline running, including the name crosswalk and the first tables in SQLite. Read the PDF tonight, and agree as a team on who is starting which part before you leave the room.

### Team progress

This cell is for your team, and nothing is submitted or graded. The only graded work in the capstone is the final submission on Friday, December 18, so these progress cells exist to help you notice early when something is behind. Fill it in and commit it with the rest of the notebook.

In [12]:
progress = {
    'acc_rows_in_torvik': len(acc),                 # 18
    'espn_date_pulled': '2026-11-13',
    'espn_games_that_day': len(events),             # 11
    'name_that_differs': 'ESPN "NC State" vs Torvik "N.C. State"',
    'blocker': 'nothing yet',
}

for k, v in progress.items():
    print(f'{k}: {v}')

acc_rows_in_torvik: 18
espn_date_pulled: 2026-11-13
espn_games_that_day: 11
name_that_differs: ESPN "NC State" vs Torvik "N.C. State"
blocker: nothing yet
